# 03 · The Causal Problem — Endogeneity & Simultaneity

This is the pivotal notebook. We have a dataset of prices and quantities and we want **one number**: the price elasticity of demand. We *know* the true value because the data is simulated — so we can catch the naive estimate red-handed.

**Why naive regression fails.** Price is *endogenous*: an unobserved **demand shock** (a heat wave, a holiday, a concert letting out) pushes price **up** and quantity **up** at the same time. Regressing $\ln Q$ on $\ln P$ then sees price and quantity moving *together* and concludes demand is barely sensitive to price — or even upward sloping. The estimate is biased toward zero / positive.

**Two fixes:**
1. **Instrumental variables / 2SLS** — find a variable (a *cost / supply shifter*) that moves price but is unrelated to the demand shock, and use only the price variation it explains.
2. **DoWhy** — state the causal graph explicitly, identify the estimand, estimate it, and **refute** it with placebo tests.

**References**
- [DoWhy documentation](https://www.pywhy.org/dowhy/)
- [linearmodels — IV2SLS](https://bashtage.github.io/linearmodels/iv/index.html)
- [Causal Inference: The Mixtape — Instrumental Variables](https://mixtape.scunning.com/07-instrumental_variables)
- [Uber — Causally-Informed Marketplace Optimization (arXiv:2407.19078)](https://arxiv.org/html/2407.19078v1)

In [1]:
import sys, os
sys.path.append(os.path.abspath('utils'))
import numpy as np
import pandas as pd
import statsmodels.api as sm
from datagen import simulate_endogenous_prices

df, gt = simulate_endogenous_prices(n=5000, true_elasticity=-1.5, confounder_strength=1.2, seed=7)
TRUE = gt['true_elasticity']
print('TRUE elasticity =', TRUE)
df.head()

TRUE elasticity = -1.5


,log_price,log_quantity,price,quantity,cost_shifter,demand_shock
0,4.175051,2.049163,65.043175,7.761399,1.115732,0.001230
1,4.356583,1.695710,77.990209,5.450517,0.624883,0.298746
2,3.298250,2.596610,27.065241,13.418171,0.428215,-0.274138
3,2.900338,2.272507,18.180293,9.703698,-0.253888,-0.890592
4,2.832765,3.154768,16.992373,23.447590,-0.504608,-0.454671


## 1. The naive (biased) regression

Regress `log_quantity` on `log_price`. The slope *should* be the elasticity. Watch it miss.

In [2]:
X = sm.add_constant(df['log_price'])
ols = sm.OLS(df['log_quantity'], X).fit()
naive = ols.params['log_price']
print(f'Naive OLS elasticity : {naive:+.3f}')
print(f'TRUE elasticity      : {TRUE:+.3f}')
print(f'Bias                 : {naive - TRUE:+.3f}  ← badly wrong, pulled toward zero')

Naive OLS elasticity : -0.849
TRUE elasticity      : -1.500
Bias                 : +0.651  ← badly wrong, pulled toward zero


## 2. Proof it's the confounder

Our simulator exposes the (normally unobservable) `demand_shock`. If we *could* control for it, OLS would be fine. This confirms the diagnosis — but in real life you can't condition on shocks you don't measure.

In [3]:
Xc = sm.add_constant(df[['log_price','demand_shock']])
ols_c = sm.OLS(df['log_quantity'], Xc).fit()
print(f'OLS controlling for the (hidden) demand shock: {ols_c.params["log_price"]:+.3f}')
print(f'TRUE elasticity                              : {TRUE:+.3f}')
print('→ With the confounder controlled, we recover the truth. The problem is purely the omitted shock.')

OLS controlling for the (hidden) demand shock: -1.510
TRUE elasticity                              : -1.500
→ With the confounder controlled, we recover the truth. The problem is purely the omitted shock.


## 3. Fix #1 — Instrumental Variables (2SLS)

The `cost_shifter` is a **supply-side instrument**: it moves price (relevance) but does not touch the demand shock (exclusion). Two-stage least squares uses only the price variation the instrument explains.

In [4]:
from linearmodels.iv import IV2SLS

data = df.assign(const=1.0)
iv = IV2SLS(dependent=data['log_quantity'], exog=data['const'],
            endog=data['log_price'], instruments=data['cost_shifter']).fit()
iv_elast = iv.params['log_price']
print(f'IV / 2SLS elasticity : {iv_elast:+.3f}')
print(f'TRUE elasticity      : {TRUE:+.3f}')

# First-stage strength check (weak instruments are dangerous).
first = sm.OLS(df['log_price'], sm.add_constant(df['cost_shifter'])).fit()
print(f'First-stage F-stat   : {first.fvalue:.0f}  (>>10 → strong instrument)')
assert abs(iv_elast - TRUE) < 0.15, iv_elast

IV / 2SLS elasticity : -1.485
TRUE elasticity      : -1.500
First-stage F-stat   : 2234  (>>10 → strong instrument)


## 4. Fix #2 — DoWhy (model → identify → estimate → refute)

DoWhy makes the assumptions explicit as a causal graph and then *stress-tests* the result. We treat the (continuous) log price as the treatment, the cost shifter as an instrument.

In [5]:
try:
    from dowhy import CausalModel
    model = CausalModel(
        data=df,
        treatment='log_price',
        outcome='log_quantity',
        instruments=['cost_shifter'],
    )
    estimand = model.identify_effect(proceed_when_unidentifiable=True)
    estimate = model.estimate_effect(estimand, method_name='iv.instrumental_variable')
    print(f'DoWhy IV estimate : {estimate.value:+.3f}   (TRUE {TRUE:+.3f})')
    assert abs(estimate.value - TRUE) < 0.15, estimate.value

    # Refutation: replacing the treatment with a permuted placebo should
    # destroy the effect (estimate ~0). IV refuters require placebo_type='permute'.
    refute = model.refute_estimate(estimand, estimate, method_name='placebo_treatment_refuter',
                                   placebo_type='permute', num_simulations=20)
    print(refute)
except Exception as e:
    print('DoWhy section skipped:', type(e).__name__, e)

DoWhy IV estimate : -1.485   (TRUE -1.500)
Refute: Use a Placebo Treatment
Estimated effect:-1.485453591686448
New effect:0.00102668504669612
p value:0.48450932666147317



## 5. Why Airbnb and Uber run experiments

Valid instruments are rare and fragile. The cleaner way to break endogeneity is to make price variation *deliberately random* — an **experiment** (A/B test, switchback, randomized price perturbation). Uber and Airbnb both lean heavily on experimentation precisely because observational price/quantity data is contaminated by the simultaneity we just demonstrated.

But experiments in a **marketplace** have their own trap: treating one rider can starve supply for another, so a naive A/B test *over-states* the true effect. That **interference / SUTVA** problem — and how to estimate heterogeneous causal price effects — is [notebook 04](04_causal_marketplace_pricing.ipynb).